# Train your own fruit-detector model

This trains a real YOLOv8 model on the fruit dataset in `dataset/` (510 real photos,
6 classes: Apple, Banana, Grape, Orange, Pineapple, Watermelon — labelled with the
Roboflow annotation tool, in standard YOLO format).

**How to use this notebook:**
1. In Colab, go to `Runtime` -> `Change runtime type` -> pick **T4 GPU** -> Save.
2. Run every cell top to bottom (the play button, or Ctrl+Enter).
3. At the end, a file called `custom_fruit.pt` downloads to your computer.
4. Drop that file into `backend/app/ml/weights/custom_fruit.pt` in your project.

Training ~510 images for 50 rounds (called "epochs") takes about 10-15 minutes on
the free GPU. You do not need to understand every line below to run it.

In [ ]:
# Step 1: install the training library
!pip install -q ultralytics

## Step 2: upload the dataset

Zip the `dataset` folder (the one next to this notebook) and upload it here.
In Colab: click the folder icon on the left, then the upload icon, and pick
`dataset.zip`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose dataset.zip when prompted

In [ ]:
import zipfile, os, shutil
with zipfile.ZipFile("dataset.zip", "r") as z:
    z.extractall("_unzipped")
# Works whether you zipped the `dataset` folder itself or just its contents.
root = "_unzipped"
if not os.path.exists(os.path.join(root, "data.yaml")):
    for name in os.listdir(root):
        if os.path.exists(os.path.join(root, name, "data.yaml")):
            root = os.path.join(root, name)
            break
shutil.rmtree("dataset", ignore_errors=True)
shutil.copytree(root, "dataset")

# Write data.yaml with an ABSOLUTE path so Ultralytics always finds the images.
yaml_text = f"""path: {os.path.abspath('dataset')}
train: train/images
val: valid/images
nc: 6
names:
  0: Apple
  1: Banana
  2: Grape
  3: Orange
  4: Pineapple
  5: Watermelon
"""
open("dataset/data.yaml", "w").write(yaml_text)
n_tr = len(os.listdir("dataset/train/images")); n_va = len(os.listdir("dataset/valid/images"))
print(f"dataset ready: {n_tr} train / {n_va} valid images")
assert n_tr > 0 and n_va > 0, "Dataset folders are empty - check your zip."

## Step 3: train

`yolov8n` is the smallest YOLOv8 model - fast to train, good enough for a
project like this. `epochs=50` means it looks at the whole dataset 50 times,
learning a bit more each pass.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # starts from a small general pretrained model,
                              # then we teach it your 6 fruit classes
results = model.train(
    data="dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="fruit_detector",
)

## Step 4: check how well it learned

This shows accuracy numbers. `mAP50` above 0.7 is solid for a small dataset
like this one.

In [ ]:
metrics = model.val()
print("mAP50    :", round(float(metrics.box.map50), 3))
print("mAP50-95 :", round(float(metrics.box.map), 3))
print("precision:", round(float(metrics.box.mp), 3), " recall:", round(float(metrics.box.mr), 3))
assert float(metrics.box.map50) > 0.5, "Model is weak - train longer (epochs=80) before using it."

## Step 5: try it on a photo

Optional - upload any photo with fruit in it and see the boxes it draws.

In [ ]:
from google.colab import files
test_upload = files.upload()  # pick a test photo, or skip this cell
import os
if test_upload:
    test_file = list(test_upload.keys())[0]
    result = model.predict(test_file, save=True, conf=0.4)
    from IPython.display import Image, display
    display(Image(filename=f"runs/detect/predict/{test_file}"))

## Step 6: download your trained model

This is the file you drop into `backend/app/ml/weights/custom_fruit.pt`.

In [ ]:
import shutil
from google.colab import files
shutil.copy("runs/detect/fruit_detector/weights/best.pt", "custom_fruit.pt")
files.download("custom_fruit.pt")  # already named correctly

## What's next

- Rename the downloaded file to `custom_fruit.pt` and put it in
  `backend/app/ml/weights/` in your project.
- In the backend, `app/ml/detector_custom.py` already knows how to load and
  run it - see `training/TRAINING.md` for the one-line switch to use it
  instead of the COCO model.
- Want more classes or better accuracy? Add more labelled images to
  `dataset/train/images` + `dataset/train/labels` (same YOLO `.txt` format)
  and re-run this notebook.